# Notebook 01 — pandas Fundamentals with Market Data

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Chapter 6 makes the point that analytics is mostly *data management*: the
modelling is the last and smallest step. This notebook is the data-management
step. By the end of it you will be able to load a raw market data file,
inspect it, reshape it, aggregate it, join reference data onto it, and handle
the gaps — which is roughly 80% of what a financial data analyst does.

**What you will learn**

| Skill | Why a finance professional needs it |
|---|---|
| `read_csv`, dtypes, dates | Vendor files arrive as text; a date stored as text cannot be sorted or differenced |
| `.loc` / `.iloc` / boolean masks | Selecting the rows a risk report actually covers |
| `groupby` | Per-desk, per-instrument, per-sector aggregation |
| `pivot` (long ↔ wide) | Market data arrives long; portfolio maths needs wide |
| `resample` | Daily marks → month-end reporting periods |
| `merge` | Attaching reference/security-master data to transactions |
| Missing-data handling | Holidays, halts, delistings, late feeds |

**Datasets used**

- `data/sp500_prices.csv` — real daily OHLCV for 30 large-cap US stocks,
  Feb 2013 – Feb 2018
- `data/sp500_companies.csv` — a security master: ticker, company name, GICS sector

## 0. Setup

Every notebook starts with this same block. It finds the repository, points at
the `data` folder, and loads the shared chart style.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "sp500_prices.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "
        "and open the mtu4600-lab3-analytics folder itself, then re-run."
    )


REPO = find_repo_root()
DATA = REPO / "data"
plt.style.use(REPO / "fin4600.mplstyle")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Repository root:", REPO)

## 1. Loading a file — and why `parse_dates` matters

First, deliberately load the file the *wrong* way, so you can see what goes
wrong. This is the single most common bug in student market-data code.

In [ ]:
wrong = pd.read_csv(DATA / "sp500_prices.csv")
print("dtype of the date column when we don't say anything:", wrong["date"].dtype)
print("\nSorting a text date is alphabetical, not chronological, which is fine")
print("for ISO dates but silently wrong for formats like 3/9/2015 vs 12/1/2014.")
print("\nAnd you cannot do arithmetic on it:")
try:
    wrong["date"].max() - wrong["date"].min()
except TypeError as exc:
    print("   TypeError:", exc)

Now load it properly. `parse_dates` converts the column to pandas'
`datetime64` type, which knows about calendars, ordering, and differences.

In [ ]:
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])

print("dtype of the date column now:", prices["date"].dtype)
print("Span of the data:", prices["date"].max() - prices["date"].min())
prices.head()

## 2. First look: shape, dtypes, describe

Before any analysis, answer three questions: *How big is it? What types are
the columns? Do the numbers look plausible?*

In [ ]:
print("Rows and columns:", prices.shape)
print()
prices.info()

`describe()` gives the five-number summary of every numeric column. Read it
sceptically — this is where you catch negative prices, zero volumes, and
prices that are off by a factor of 100 because someone stored cents.

In [ ]:
prices.describe()

## 3. Selecting rows and columns

Three tools, and it is worth being clear about when to use each:

- `df["col"]` — one column, by name
- `df.loc[rows, cols]` — select by **label** (a ticker, a date, a column name)
- `df.iloc[rows, cols]` — select by **integer position** (row 0, row 5)

In practice, `.loc` with a boolean mask is what you will use 90% of the time.

In [ ]:
# A single column is a Series
print(type(prices["close"]))

# A list of columns is a DataFrame
prices[["date", "ticker", "close"]].head(3)

In [ ]:
# Boolean mask: one bank's rows only
jpm = prices.loc[prices["ticker"] == "JPM"]
print("JPM rows:", len(jpm))

# Two conditions. Note the parentheses — & and | bind tighter than == in Python,
# so leaving the parentheses out is a syntax error.
big_down_days = prices.loc[
    (prices["ticker"] == "JPM") & (prices["close"] < prices["open"] * 0.97)
]
print("JPM days that closed more than 3% below the open:", len(big_down_days))
big_down_days[["date", "open", "close", "volume"]]

In [ ]:
# .isin() for a set of labels
banks = prices.loc[prices["ticker"].isin(["JPM", "GS", "BAC", "WFC"])]
print("Bank rows:", len(banks), "across", banks["ticker"].nunique(), "tickers")

### Exercise 1

Select every row for `XOM` (Exxon) in calendar year 2015 where daily volume
was above 20 million shares. How many such days were there?

*Hint:* `prices["date"].dt.year` gives the year of a datetime column.

In [ ]:
# YOUR CODE HERE
xom_2015_high_volume = prices.loc[
    (prices["ticker"] == "XOM")
    & (prices["date"].dt.year == 2015)
    & (prices["volume"] > 20_000_000)
]

len(xom_2015_high_volume)  # 24

## 4. Adding computed columns

Never write a Python loop over rows. pandas operations are *vectorised* —
they apply to the whole column at once, in compiled code. On this dataset the
difference is roughly a hundredfold, and on a real tick file it is the
difference between a report that runs and one that does not.

In [ ]:
prices["daily_range_pct"] = (prices["high"] - prices["low"]) / prices["close"] * 100
prices["dollar_volume"] = prices["close"] * prices["volume"]
prices["year"] = prices["date"].dt.year

prices[["date", "ticker", "close", "daily_range_pct", "dollar_volume", "year"]].head()

## 5. `groupby` — the workhorse

"Split, apply, combine". Split the rows into groups, apply a function to each
group, combine the answers into one table. Every per-desk, per-sector, or
per-instrument report you will ever build is a `groupby`.

In [ ]:
# One statistic per ticker
by_ticker = prices.groupby("ticker")["close"].mean().sort_values(ascending=False)
by_ticker.head(8)

In [ ]:
# Several statistics at once, with readable output column names
summary = prices.groupby("ticker").agg(
    first_close=("close", "first"),
    last_close=("close", "last"),
    avg_daily_range_pct=("daily_range_pct", "mean"),
    avg_dollar_volume_mm=("dollar_volume", lambda s: s.mean() / 1e6),
    n_days=("close", "size"),
)
summary["total_return_pct"] = (
    summary["last_close"] / summary["first_close"] - 1
) * 100

summary.sort_values("total_return_pct", ascending=False).round(2).head(10)

In [ ]:
# Group by two keys — the result has a MultiIndex
by_ticker_year = prices.groupby(["ticker", "year"])["dollar_volume"].mean()
by_ticker_year.head(6)

### Exercise 2

Which three tickers had the **highest average daily range** (`daily_range_pct`)
over the whole period? Does the answer match your intuition about which of
these businesses is riskiest?

In [ ]:
# YOUR CODE HERE

## 6. Long vs wide: `pivot`

The file is in **long** format — one row per ticker per day. That is how
market data is stored and transmitted, because it handles instruments with
different histories cleanly.

Portfolio mathematics needs **wide** format — one row per day, one column per
ticker. Correlation matrices, covariance matrices and portfolio returns all
assume a wide matrix. Converting between the two is a daily task.

In [ ]:
close_wide = prices.pivot(index="date", columns="ticker", values="close")

print("Long format:", prices.shape)
print("Wide format:", close_wide.shape)
close_wide.iloc[:5, :6]

In [ ]:
# ...and back again, which you need when writing to a database
back_to_long = close_wide.stack().rename("close").reset_index()
back_to_long.head(3)

## 7. `resample` — changing the time frequency

Daily marks, month-end reporting. `resample` only works on a datetime index,
which is why we pivoted first.

The rule strings you will use most: `"ME"` month-end, `"QE"` quarter-end,
`"YE"` year-end, `"W"` weekly.

In [ ]:
monthly_close = close_wide.resample("ME").last()
print("Daily observations:", len(close_wide))
print("Monthly observations:", len(monthly_close))
monthly_close.iloc[:5, :6].round(2)

In [ ]:
# Different columns often want different rules. Volume sums; price takes the last.
aapl_daily = prices.loc[prices["ticker"] == "AAPL"].set_index("date")
aapl_monthly = aapl_daily.resample("ME").agg(
    open=("open", "first"),
    high=("high", "max"),
    low=("low", "min"),
    close=("close", "last"),
    volume=("volume", "sum"),
)
aapl_monthly.head().round(2)

## 8. Joining reference data: `merge`

Prices alone cannot answer "how did Financials do?". For that you need a
**security master** — the reference table that maps an instrument identifier
to its attributes. Every bank has one, and keeping it correct is a permanent
operational headache.

In [ ]:
companies = pd.read_csv(DATA / "sp500_companies.csv")
print("Security master rows:", len(companies))
companies.head()

### The join that quietly loses rows

An **inner** join keeps only rows that match on both sides. A **left** join
keeps every price row and leaves `NaN` where the master has no entry.

In production you almost always want a left join *plus an alert*, because a
silent inner join is how a position disappears from a risk report.

In [ ]:
inner = prices.merge(companies, on="ticker", how="inner")
left = prices.merge(companies, on="ticker", how="left")

print("Price rows before the join:", len(prices))
print("After an inner join:       ", len(inner))
print("After a left join:         ", len(left))

unmatched = left.loc[left["sector"].isna(), "ticker"].unique()
print("\nTickers with no security-master entry:", list(unmatched) or "none")

Here every ticker matched. That is because the file was curated for this lab.
Reality is messier: the security master is a *current* index membership list,
while the price history covers 2013–2018, so a real join of the full files
would strand every company that has since been acquired or removed. Let's
simulate that, because recognising the symptom is the skill.

In [ ]:
# Pretend the security master lost three entries, as if those firms were acquired
damaged = companies.loc[~companies["ticker"].isin(["GE", "T", "DUK"])]

risky_inner = prices.merge(damaged, on="ticker", how="inner")
safe_left = prices.merge(damaged, on="ticker", how="left")

print(f"Inner join silently dropped {len(prices) - len(risky_inner):,} rows.")
print("Left join kept them all and flagged the problem:")
print("   unmatched tickers:", sorted(safe_left.loc[safe_left['sector'].isna(), 'ticker'].unique()))

**Rule of thumb:** after every join, compare the row count to what you
expected. A join that changes the row count unexpectedly is a bug until
proven otherwise.

In [ ]:
# Now the sector question is answerable
sector_stats = (
    left.groupby("sector")
    .agg(
        tickers=("ticker", "nunique"),
        avg_daily_range_pct=("daily_range_pct", "mean"),
        avg_dollar_volume_mm=("dollar_volume", lambda s: s.mean() / 1e6),
    )
    .sort_values("avg_daily_range_pct", ascending=False)
)
sector_stats.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_data = sector_stats["avg_daily_range_pct"].sort_values()
ax.barh(plot_data.index, plot_data.values, color="#2a78d6", height=0.65)
ax.set_title("Average daily high–low range by GICS sector, 2013–2018")
ax.set_xlabel("Daily range as % of closing price")
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
for y, v in enumerate(plot_data.values):
    ax.text(v + 0.02, y, f"{v:.2f}%", va="center", fontsize=9, color="#52514e")
plt.show()

## 9. Missing data

Markets close. Stocks halt. Feeds arrive late. Companies delist. Your data
will have holes, and how you fill them changes your answer.

In [ ]:
# The trading calendar is not the calendar calendar
all_days = pd.date_range(close_wide.index.min(), close_wide.index.max(), freq="D")
business_days = pd.date_range(close_wide.index.min(), close_wide.index.max(), freq="B")

print("Calendar days in the period: ", len(all_days))
print("Business days:               ", len(business_days))
print("Actual trading days in data: ", len(close_wide))
print("Difference (market holidays):", len(business_days) - len(close_wide))

In [ ]:
# Reindexing onto every business day exposes the holidays as NaN
reindexed = close_wide.reindex(business_days)
print("Missing values introduced:", int(reindexed.isna().sum().sum()))

holidays = reindexed.index[reindexed["AAPL"].isna()]
print("\nFirst eight market holidays in the sample:")
for d in holidays[:8]:
    print("   ", d.date(), d.day_name())

Three ways to deal with a hole, and they are not interchangeable:

| Method | What it does | When it is right |
|---|---|---|
| `.dropna()` | delete the row | the observation genuinely does not exist |
| `.ffill()` | carry the last price forward | a market holiday — the position still exists, the price just did not change |
| `.interpolate()` | straight-line between neighbours | almost never for prices; it invents a price that never traded |

**Forward fill is right for a holiday. Interpolation is not**, because it uses
tomorrow's price to fill in today — information you would not have had. That
is *look-ahead bias*, and it is the defect that makes most amateur backtests
worthless. Notebook 03 returns to this.

### A real example: Chevron, Thanksgiving 2014

US markets were closed on Thursday 27 November 2014. That same day, OPEC met
in Vienna and declined to cut production. When the market reopened on the
Friday, oil and the energy sector fell hard.

Look at what each filling method puts in the Thursday hole.

In [ ]:
comparison = pd.DataFrame(
    {
        "raw": reindexed["CVX"],
        "ffill": reindexed["CVX"].ffill(),
        "interpolate": reindexed["CVX"].interpolate(),
    }
)
comparison.loc["2014-11-24":"2014-12-01"].round(2)

Forward fill puts **115.11** in the Thursday hole — the last price that
actually traded, which is what a position would have been marked at.

Interpolation puts roughly **112** there, a number that is part-way to
Friday's post-OPEC price. Nobody could have known that on Thursday. If you
then computed a Thursday-to-Friday return from the interpolated series, your
model would appear to have anticipated the OPEC announcement — and your
backtest would show skill that does not exist.

### Exercise 3

Count how many trading days each ticker actually has in `close_wide`. Are
they all the same? What would a ticker with fewer days tell you about that
company's history?

*Hint:* `close_wide.notna().sum()`

In [ ]:
# YOUR CODE HERE

## 10. Saving your work

Save the wide price matrix — Notebook 02 picks it up from here.

In [ ]:
processed = REPO / "data" / "processed"
processed.mkdir(exist_ok=True)

close_wide.to_csv(processed / "close_wide.csv")
print("Wrote", processed / "close_wide.csv")
print(f"{close_wide.shape[0]} rows x {close_wide.shape[1]} columns")

---

## Recap

1. Parse dates on load, or nothing downstream works.
2. `describe()` before you model — it catches bad data cheaply.
3. `groupby` is split-apply-combine; it answers nearly every "by desk / by
   sector / by instrument" question.
4. Market data is stored long and analysed wide. `pivot` converts.
5. Check the row count after every `merge`.
6. Forward-fill holidays. Do not interpolate prices.

## Your turn

Complete the three exercises above, then commit and push:

```
git add notebooks/01_pandas_fundamentals.ipynb
git commit -m "Completed notebook 01 exercises"
git push
```

Next: `02_returns_and_risk.ipynb`